# 01 Quick Start

Create a compact playground with the main nbplay widgets. Run the cell, then open the tabs and interact with each surface.


In [ ]:
import ipywidgets as widgets
import traitlets

from nbplay import (
    KeyboardWidget,
    MidiKeyboardWidget,
    MixerWidget,
    PadWidget,
    SamplerWidget,
    SequencerWidget,
    SettingsWidget,
    SynthWidget,
    TimelineWidget,
    TransportWidget,
    EffectPlugin,
)

settings = SettingsWidget()
transport = TransportWidget(bpm=120.0)

synth = SynthWidget(oscillator_type="saw", frequency=440.0, amplitude=0.35)
sampler = SamplerWidget(sample_name="Drop audio here", pad_count=8)
seq = SequencerWidget(length=8, num_voices=2, bpm=120.0)
for i, note in enumerate([60, 64, 67, 72, 67, 64, 62, 59]):
    seq.set_step(i, note=note, velocity=96, active=i % 2 == 0, voice=0)

keyboard = KeyboardWidget(upper_octave=4, lower_octave=3, velocity=96)
midi_keyboard = MidiKeyboardWidget(upper_octave=4, lower_octave=3)
pads = PadWidget(rows=2, cols=4, velocity=110)

mixer = MixerWidget()
channel = mixer.add_channel("Preview")
mixer.add_channel_effect(channel, EffectPlugin("compressor", threshold=-18, ratio=4))
mixer.add_master_effect(EffectPlugin("limiter", threshold=-1))

timeline = TimelineWidget(
    bpm=transport.bpm,
    length=64,
    count_in_bars=1,
    auto_extend_recording=True,
    recording_extend_bars=16,
)
transport_links = [
    traitlets.link((transport, "bpm"), (timeline, "bpm")),
    traitlets.link((transport, "time_signature_num"), (timeline, "time_signature_num")),
    traitlets.link((transport, "time_signature_den"), (timeline, "time_signature_den")),
    traitlets.link((transport, "is_playing"), (timeline, "is_playing")),
    traitlets.link((transport, "is_recording"), (timeline, "is_recording")),
    traitlets.link((transport, "current_beat"), (timeline, "current_beat")),
]
timeline.add_track("Audio", channel_index=channel, armed=True, monitor=True)
timeline.add_clip("Drop or record here", track_index=0, start=0, duration=4, source="placeholder")

tabs = widgets.Tab([
    widgets.VBox([settings, transport]),
    widgets.VBox([synth, seq]),
    widgets.VBox([sampler, pads]),
    widgets.VBox([keyboard, midi_keyboard]),
    widgets.VBox([timeline, mixer]),
])
for index, title in enumerate(["Setup", "Synth + Seq", "Sampler + Pads", "Keys", "Timeline + Mixer"]):
    tabs.set_title(index, title)

tabs
